# Milestone 4 — Transformer training (Experiments C and D)

**Runs on either Google Colab or Kaggle.** Full instructions, secrets and runtime
estimate: `research/scripts/MILESTONE_4_GPU_HANDOFF.md`.

Trains **mBERT (C)** and **XLM-R-base (D)** in-domain on Ax-to-Grind, 3 seeds each
(`{42, 123, 2026}`), writes metrics through the project's existing metrics contract,
and pushes each checkpoint to a **staging** HF repo.

One notebook, both platforms — not two copies that drift. The only differences are
the secret store, the working directory and how the metrics zip comes back; all
three live in `research/src/notebook_env.py`. The training itself is identical.

## Before you start

### On Kaggle

1. **Settings → Accelerator → GPU T4 x2** (or P100). Kaggle has no
   `Runtime → Change runtime type` menu — the accelerator is a notebook setting.
2. **Settings → Internet → On.** ⚠️ Kaggle disables internet **by default**, and
   with it off the repo clone, `pip install` and the HF checkpoint push all fail.
   This is the easiest blocker to miss. (Requires a phone-verified Kaggle account.)
3. **Add-ons → Secrets** → add `HF_TOKEN` (your HF **write** token) and make sure
   it is **attached to this notebook**.
4. Set `HF_STAGING_PREFIX` in the config cell below to your HF username.

### On Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Add your HF **write** token to Colab **Secrets** (🔑 sidebar) as `HF_TOKEN`,
   with notebook access on.
3. Set `HF_STAGING_PREFIX` in the config cell below to your HF username.

Colab has internet on always; there is no toggle to set.

## 1. Pre-flight — GPU and internet

Stops immediately if there is no GPU, rather than silently spending hours on CPU.

Also checks outbound internet, because on **Kaggle it is off by default**. Without
it the clone, the `pip install` and the checkpoint push all fail — several cells
apart, with three unrelated-looking errors. Checking here makes it one message.

In [ ]:
import os, socket, subprocess, sys

# Which host is this? This cell runs BEFORE the repo is cloned, so it cannot import
# research/src/notebook_env.py yet — this is the one place the detection is
# duplicated, deliberately, and only the wording of messages depends on it.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle")

# --- GPU -------------------------------------------------------------------
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit(
        "No GPU detected. "
        + (
            "Settings -> Accelerator -> GPU T4 x2 (or P100), then rerun."
            if ON_KAGGLE
            else "Runtime -> Change runtime type -> T4 GPU, then rerun."
        )
    )
print(out.stdout)

# --- Internet --------------------------------------------------------------
# Kaggle notebooks have internet DISABLED by default; git clone, pip install and
# the HF push all need it.
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
    print("internet: OK")
except OSError as exc:
    sys.exit(
        f"No outbound internet ({exc}).\n"
        + (
            "On Kaggle this is the DEFAULT. Settings -> Internet -> On (right-hand "
            "panel; needs a phone-verified account), then rerun from the top."
            if ON_KAGGLE
            else "Colab normally has internet — check the connection and rerun."
        )
    )

# --- Pin to ONE GPU --------------------------------------------------------
# Kaggle's default accelerator is "GPU T4 x2". With two GPUs visible, the HF
# Trainer transparently wraps the model in nn.DataParallel and treats
# per_device_train_batch_size as PER DEVICE -- so the configured 16 becomes an
# effective batch of 32, and the 401 optimizer steps/epoch this project's runtime
# estimate and 3-seed design are built on become ~201. That is a silent change to
# the training dynamics, not a speed-up.
#
# The research design is frozen (config unchanged, batch size unchanged); making
# only one GPU visible is what KEEPS it identical to the single-T4 Colab run this
# was written for. Set before torch is imported anywhere, and inherited by the
# `!python -m ...` training subprocesses below.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES=0 (single GPU, matching the configured batch size)")

print("platform:", "Kaggle" if ON_KAGGLE else "Colab / other")

## 2. Configuration

`HF_STAGING_PREFIX` is not secret — set it here. `HF_TOKEN` is read from the
platform's secret store (Colab Secrets / Kaggle Add-ons → Secrets) in section 3,
once the repo that knows how to read it is present.

In [ ]:
# ---- EDIT THIS ----
HF_STAGING_PREFIX = "your-hf-username"   # e.g. "rehman-ayoub"
REPO_URL = "https://github.com/<your-github-username>/<repo-name>.git"
REPO_BRANCH = "main"
# -------------------

# Where the clone goes. /content exists only on Colab; on Kaggle the writable,
# roomy, session-persisted directory is /kaggle/working. Once the repo is synced,
# research/src/notebook_env.py owns this mapping (repo_dir()) — it cannot be
# imported yet, because it lives in the repo about to be cloned.
REPO_DIR = "/kaggle/working/repo" if ON_KAGGLE else "/content/repo"

# Identifies THIS notebook document. Bumped whenever a cell below changes, and
# checked against the committed copy after the repo is synced (section 3).
# Restarting the runtime restarts the kernel but does NOT reload the notebook
# source, so an open tab can quietly keep running pre-fix cells against a freshly
# updated repo — which is exactly how the 2026-08-15 run failed. This turns that
# into a clear error. Applies equally to Kaggle's "Factory reset".
NOTEBOOK_REVISION = 4

assert HF_STAGING_PREFIX != "your-hf-username", "Set HF_STAGING_PREFIX to your HF username first."
print("staging namespace:", HF_STAGING_PREFIX)
print("repo directory:   ", REPO_DIR)
print("notebook revision:", NOTEBOOK_REVISION)

## 3. Sync the repo, read the secret, install pinned dependencies

The first cell **re-syncs to the branch tip every run**, not only on a first clone,
and prints the commit it landed on. The second checks that the notebook you are
running is not older than the one in that commit — restarting the runtime reloads
the kernel but *not* the notebook source, so an open tab can otherwise keep
executing pre-fix cells against fixed repo code.

The third reads `HF_TOKEN`, and comes **after** the sync on purpose: it uses
`research/src/notebook_env.py`, which lives in the repo and is what lets this one
notebook run on both platforms. That module is stdlib-only, so it imports fine
here — before `pip install` has run.

In [ ]:
import os, subprocess, sys

# Sync to the branch tip on EVERY run, not just when the directory is missing.
#
# The previous version cloned only if the directory was absent. Restarting the
# session keeps the working directory, so on any rerun the clone was skipped and the
# repo silently stayed at whatever commit it was first cloned at — no output said so.
# Two sources of truth (notebook cells vs repo code) then drift apart with nothing
# reporting it. `checkout -B` resets the local branch onto origin's tip, which is
# idempotent and safe here: research/data/raw/ is gitignored, and untracked files
# (downloaded corpora, produced metrics) are left alone by a hard branch reset.
#
# Plain git throughout. Nothing in this cell is platform-specific beyond REPO_DIR,
# so it behaves identically on Colab and Kaggle — but Kaggle needs internet ENABLED
# for the clone and fetch to work at all (checked in the pre-flight cell).
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
subprocess.run(
    ["git", "checkout", "--quiet", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True
)

# `import research.src...` needs the repo root on sys.path. Colab's IPython puts the
# CWD there implicitly; Kaggle's starts in /kaggle/working and does not.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("repo synced to:")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Stale-notebook guard. Compares the notebook YOU are running against the copy just
# synced from the repo, and stops if yours is older.
#
# Why this exists: on 2026-08-15 a run had a correctly-updated repo but a pre-fix
# notebook tab. The cells issued old commands against new code, so the data check ran
# unscoped and the smoke test hit the torchvision error the new cells prevent. Nothing
# reported the mismatch; it looked like the fixes had simply not worked.
import json, re

with open("research/notebooks/04_transformer_training.ipynb", encoding="utf-8") as fh:
    committed = json.load(fh)

marker = re.compile(r"NOTEBOOK" + r"_REVISION\s*=\s*(\d+)")
found = [
    int(m.group(1))
    for cell in committed["cells"]
    for m in marker.finditer("".join(cell["source"]))
]
repo_revision = max(found) if found else 0

if repo_revision > NOTEBOOK_REVISION:
    raise SystemExit(
        f"STALE NOTEBOOK — you are running revision {NOTEBOOK_REVISION}, the repo has "
        f"revision {repo_revision}.\n\n"
        "Restarting the runtime restarts the kernel; it does NOT reload the notebook "
        "source in an already-open tab.\n"
        "Fix: close this tab, re-open the notebook from GitHub "
        "(File -> Open notebook -> GitHub tab -> this repo), and run from the top."
    )

print(f"notebook revision {NOTEBOOK_REVISION}, repo revision {repo_revision} — in sync")

In [ ]:
import os

# Read the write token from the platform's secret store, so it never appears in
# saved notebook output. notebook_env picks the right one:
#   Colab  -> google.colab.userdata            (Secrets, key icon, left sidebar)
#   Kaggle -> kaggle_secrets.UserSecretsClient (Add-ons -> Secrets)
# and raises with the platform-appropriate fix if the secret is missing, empty, or
# the host is neither. Unit-tested in research/tests/test_notebook_env.py.
#
# Imported before the pip install below, which is safe: notebook_env is stdlib-only.
from research.src.notebook_env import detect_platform, get_secret

print("detected platform:", detect_platform())

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
os.environ["HF_STAGING_PREFIX"] = HF_STAGING_PREFIX

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("HF_STAGING_PREFIX set:", os.environ["HF_STAGING_PREFIX"])

In [ ]:
# Step 1 of 2 — remove the image's preinstalled torchvision/torchaudio BEFORE
# installing. Applies to Kaggle as much as to Colab: Kaggle's notebook image is
# built FROM the Colab runtime image, so it ships the same preinstalled copies.
#
# They are compiled against whatever torch the image shipped with. The install below
# moves torch to this project's pin, and the leftovers then register C++ ops against
# the wrong ABI, so `from transformers import Trainer` dies with
#     RuntimeError: operator torchvision::nms does not exist
# transformers guards that import with is_torchvision_available(), which only checks
# whether the package is INSTALLED, not whether it imports — a present-but-broken
# copy passes the guard and raises RuntimeError, which nothing on that path catches.
# With torchvision absent the guard is simply False and the block is skipped.
# This project does zero vision and zero audio work. See research/requirements.txt.
#
# pip will warn that fastai/timm now have an unsatisfied torchvision requirement.
# That is expected and harmless — nothing in this pipeline imports them.
!pip uninstall -y -q torchvision torchaudio

# Step 2 of 2 — install the pinned set (REPRODUCIBILITY.md Section 1). On Linux
# (both platforms are Linux) the plain torch pin resolves to the CUDA build,
# which is what a T4 or a P100 needs.
!pip install -q -r research/requirements.txt

In [ ]:
# Environment gate. Fails here, in two seconds, rather than an hour into training.
import importlib.util

import torch

assert importlib.util.find_spec("torchvision") is None, (
    "torchvision is still installed. Re-run the cell above; if it persists, "
    "restart the session (Colab: Runtime -> Restart session; Kaggle: "
    "Run -> Restart session) and run this notebook from the top."
)

# The exact import that failed on the first Colab attempt. Kept as an assertion so a
# future environment drift is caught by this cell and not by the smoke test.
from transformers import Trainer  # noqa: F401
import transformers

# The pins from research/requirements.txt, asserted rather than merely printed --
# printing a drifted version does not stop the run, and the whole point of this cell
# is to stop it. Kept in sync with that file by hand; it is the only place they are
# duplicated, and the assertion is what makes a mismatch loud instead of silent.
EXPECTED_TORCH = "2.13.0"
EXPECTED_TRANSFORMERS = "5.15.0"

# torch appends a local build tag: "2.13.0+cu130" on a Colab GPU, "2.13.0+cpu" on the
# authoring box. Same pinned version, different accelerator, so compare the release
# part only -- an equality check against the raw string would fail on every machine.
torch_version = torch.__version__.split("+")[0]

assert torch_version == EXPECTED_TORCH, (
    f"torch is {torch.__version__}, expected {EXPECTED_TORCH}. The pinned install did "
    "not take effect -- re-run the install cell above."
)
assert transformers.__version__ == EXPECTED_TRANSFORMERS, (
    f"transformers is {transformers.__version__}, expected {EXPECTED_TRANSFORMERS}. "
    "The pinned install did not take effect -- re-run the install cell above."
)

print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__, "| Trainer import OK")
print("python      ", __import__("sys").version.split()[0])
print("version pins OK:", f"torch=={EXPECTED_TORCH}, transformers=={EXPECTED_TRANSFORMERS}")
# Python 3.12 exactly, per REPRODUCIBILITY.md Section 1 / DECISION_REGISTER.md M4-2.
# Colab and Kaggle both provide it (Kaggle builds FROM the Colab image). Asserted
# rather than printed because M4-2's whole point is that every committed number
# comes from ONE interpreter -- a silent drift here would quietly undo that.
py_version = ".".join(__import__("sys").version.split()[0].split(".")[:2])
assert py_version == "3.12", (
    f"Python is {py_version}, expected 3.12 (REPRODUCIBILITY.md Section 1). The\n"
    "classical baselines were computed on 3.12; running the transformers on a\n"
    "different interpreter reintroduces exactly the ambiguity M4-2 removed.\n"
    "Do not work around this -- report it back so it can be logged."
)

assert torch.cuda.is_available(), (
    "torch cannot see the GPU -- check the accelerator setting "
    "(Kaggle: Settings -> Accelerator; Colab: Runtime -> Change runtime type)."
)

## 4. Data check — Ax-to-Grind only

Training reads the **committed split index files**, so the raw corpus must be
present. `research/data/raw/` is gitignored, so this re-downloads it (no
credentials needed).

Scoped to Ax-to-Grind deliberately: Experiments C and D train and evaluate on it
alone. Notri-Fact is the held-out **cross-dataset** test set and is not read until
Milestone 5 (`EXPERIMENT_PLAN.md` step 4), so requiring it here would fail the run
over a file this milestone never opens. The full-coverage checks stay exactly as
they are for CI and the complete pipeline — this cell selects from them, it does
not replace them.

In [ ]:
!python -m research.src.data.download --only ax_to_grind

# download.py regenerates MANIFEST.sha256 wholesale from whatever is on disk, so
# after a single-dataset download it holds hashes recomputed from the files just
# fetched — verifying those files against it would be circular and would prove
# nothing. Restore the COMMITTED manifest, which is the actual dataset-version
# anchor (REPRODUCIBILITY.md Section 3), before checking anything against it.
!git checkout -- research/data/raw/MANIFEST.sha256

# Presence + checksum, Ax-to-Grind only. `-k ax_to_grind` selects
# test_manifest_covers_ax_to_grind plus the three per-file checksum cases; the
# Notri-Fact and whole-manifest cases stay in the file untouched for CI.
!python -m pytest research/tests/test_raw_data_integrity.py -k ax_to_grind -q

# Schema, label and row-count validation, Ax-to-Grind only.
!python -m research.src.data.validate --only ax_to_grind

## 5. Smoke test — a few real steps before the real job

**If this fails, stop and send the error.** It costs a minute and catches setup
problems before an hour of GPU time is spent on them.

In [ ]:
!python -m research.src.models.transformer --dry-run --experiments D --dry-run-max-length 128

## 6. The real runs — 2 models × 3 seeds

~40–80 min total. Checkpoints push to staging as each seed finishes, so an
interrupted session does not lose completed work.

If you would rather split across two sessions, run the two cells separately.

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments C

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments D

## 7. Summary — copy this table back

In [ ]:
import glob, json

rows = []
for path in sorted(glob.glob("research/results/metrics/[CD]_*.json")):
    d = json.load(open(path, encoding="utf-8"))
    rows.append((
        d["experiment_id"], d["model"], d["split"], d["seed"],
        round(d["metrics"]["macro_f1"], 4),
        round(d["metrics"]["accuracy"], 4),
        d["prediction_collapse"]["is_collapsed"],
        d["run_metadata"].get("truncation", {}).get("pct_truncated"),
        round(d["run_metadata"].get("train_runtime_seconds", 0) / 60, 1),
    ))

hdr = f"{'exp':<4}{'model':<34}{'split':<6}{'seed':<7}{'macroF1':>9}{'acc':>8}{'collapsed':>11}{'trunc%':>8}{'min':>7}"
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r[0]:<4}{r[1]:<34}{r[2]:<6}{r[3]:<7}{r[4]:>9}{r[5]:>8}{str(r[6]):>11}{str(r[7]):>8}{r[8]:>7}")

print("\nSeed variance (test macro-F1) — report this, EXPERIMENT_PLAN.md Section 5:")
import statistics
for exp in ("C", "D"):
    vals = [r[4] for r in rows if r[0] == exp and r[2] == "test"]
    if len(vals) > 1:
        print(f"  {exp}: mean={statistics.mean(vals):.4f} sd={statistics.stdev(vals):.4f} values={vals}")

## 8. Package the metrics for download

Bring the zip back, plus the staging repo IDs and **revision SHAs** printed above
(`REPRODUCIBILITY.md` Section 6 needs repo ID *and* revision — a branch name moves,
a SHA does not).

On Colab this triggers a browser download. **Kaggle has no such call**: the file is
written into `/kaggle/working` and you collect it from the session's **Output** tab.

In [ ]:
import zipfile
from pathlib import Path

from research.src.notebook_env import deliver_file, working_root

# Written into the platform's working root: /content on Colab, /kaggle/working on
# Kaggle — on Kaggle it is the Output tab that hands the file back, so a zip written
# anywhere else there is one nobody can retrieve.
archive = working_root() / "milestone4_metrics.zip"

# [CD]_*.json only — this milestone's output. The same directory also holds the
# committed Milestone 3 baselines (A, B, H, H2), which are already in git and do not
# need carrying back.
metrics = sorted(Path("research/results/metrics").glob("[CD]_*.json"))
assert metrics, "no C/D metrics files found — did the training cells actually run?"

# zipfile rather than `!zip`: the zip binary is not guaranteed on either image, and
# when it is missing the shell form fails quietly and leaves no archive behind.
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in metrics:
        zf.write(path, arcname=path.name)

print(f"zipped {len(metrics)} files -> {archive}")
print(deliver_file(archive))